# Report vytíženosti pracovišť (Back Office)

Tento notebook spočítá vytíženost jednotlivých pracovišť a poboček vůči jejich denní kapacitě
a vygeneruje samostatný interaktivní **HTML report**.

**Vstupní soubory:**
- `bo_data.xlsx` — log aktivit: `BRANCH_ID, PRACOVISTE_ID, DATETIME, ZAMESTNANEC, ACTIVITY, DURATION` (minuty)
- `work_spaces.xlsx` — kapacity poboček: `BRANCH_ID, BRANCH_NAME, NO_WPL` (počet pracovišť), `CAPACITY` (kapacita 1 pracoviště v hod/den)

**Postup:** nahraďte cesty v buňce *Konfigurace* svými soubory (nebo je nahrajte do `data/`) a spusťte notebook
odshora dolů (`Kernel → Restart & Run All`). Pro rychlý test beze změny cest je připraven ukázkový
vzorek dat v `data/sample/` — přesně data ze zadání.

In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))

import report_lib as rl
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

print("Verze report_lib.py:", getattr(rl, "__version__", "neznámá"))


Verze report_lib.py: 2026-07-03c (diagnostika NaT / neshoda BRANCH_ID)


## 1. Konfigurace

- `BO_DATA_FILE` / `WORKSPACES_FILE`: cesty ke vstupním souborům.
- `rl.BUSINESS_DAYS_ONLY`: `True` = vytíženost se počítá jen pro pracovní dny Po–Pá (typický provoz pobočky).
- `rl.THRESHOLD_HIGH` / `rl.THRESHOLD_CRITICAL`: hranice (v %) pro barevné rozlišení „Vysoká“ / „Kritická“ vytíženost.

In [2]:
# Pro test s ukázkovými daty ze zadání ponechte "data/sample/...".
# Pro ostré vyhodnocení nahraďte cestami k reálným souborům, např. "data/bo_data.xlsx".
BO_DATA_FILE = "../data/sample/bo_data.xlsx"
WORKSPACES_FILE = "../data/sample/work_spaces.xlsx"

OUTPUT_HTML = "../output/vytizenost_report.html"

rl.BUSINESS_DAYS_ONLY = True
rl.THRESHOLD_HIGH = 70.0
rl.THRESHOLD_CRITICAL = 90.0

## 2. Načtení dat

In [3]:
activities, data_issues = rl.load_activities(BO_DATA_FILE)
workspaces = rl.load_workspaces(WORKSPACES_FILE)

print(f"Aktivity: {len(activities)} platných řádků, {len(data_issues)} přeskočeno (chybná data).")
print(f"Pobočky (work_spaces.xlsx): {len(workspaces)}")
activities.head()

Aktivity: 14 platných řádků, 0 přeskočeno (chybná data).
Pobočky (work_spaces.xlsx): 18


,BRANCH_ID,WORKSTATION_ID,DATETIME_RAW,EMPLOYEE,ACTIVITY,DURATION_MIN,DATETIME,DATE,END_DATETIME
0,459,5,02.07.2026 09:25:40,Škábová Alena,Online schůzka s klientem,51,2026-07-02 09:25:40,2026-07-02,2026-07-02 10:16:40
1,459,18,02.07.2026 09:44:05,Radil Ivo,Aktivita bez klienta,60,2026-07-02 09:44:05,2026-07-02,2026-07-02 10:44:05
2,459,13,02.07.2026 10:56:40,Szücs Magdalena,Online schůzka s klientem,14,2026-07-02 10:56:40,2026-07-02,2026-07-02 11:10:40
3,459,6,02.07.2026 11:41:50,Tauchman Dominik,Online schůzka s klientem,46,2026-07-02 11:41:50,2026-07-02,2026-07-02 12:27:50
4,459,18,02.07.2026 12:43:56,Radil Ivo,Online schůzka s klientem,60,2026-07-02 12:43:56,2026-07-02,2026-07-02 13:43:56


In [4]:
if not data_issues.empty:
    display(data_issues)

In [5]:
merged, unknown_branches = rl.merge_activities_with_workspaces(activities, workspaces)

if not unknown_branches.empty:
    print("Pozor: aktivity patří pobočkám, které nejsou ve work_spaces.xlsx:")
    display(unknown_branches[["BRANCH_ID", "WORKSTATION_ID", "DATETIME", "EMPLOYEE"]])

if merged.empty:
    print("BRANCH_ID v bo_data.xlsx:      ", sorted(activities["BRANCH_ID"].unique()))
    print("BRANCH_ID v work_spaces.xlsx:  ", sorted(workspaces["BRANCH_ID"].unique()))
    raise ValueError(
        "Po spojení s work_spaces.xlsx nezůstala žádná platná aktivita (žádné BRANCH_ID se "
        "neshoduje mezi bo_data.xlsx a work_spaces.xlsx) — viz vypsané seznamy ID výš."
    )

merged.head()

,BRANCH_ID,WORKSTATION_ID,DATETIME_RAW,EMPLOYEE,ACTIVITY,DURATION_MIN,DATETIME,DATE,END_DATETIME,BRANCH_NAME,NO_WORKSTATIONS,CAPACITY_HOURS,CAPACITY_MIN
0,459,5,02.07.2026 09:25:40,Škábová Alena,Online schůzka s klientem,51,2026-07-02 09:25:40,2026-07-02,2026-07-02 10:16:40,Praha 2 (Jugoslávská),20,8,480
1,459,18,02.07.2026 09:44:05,Radil Ivo,Aktivita bez klienta,60,2026-07-02 09:44:05,2026-07-02,2026-07-02 10:44:05,Praha 2 (Jugoslávská),20,8,480
2,459,13,02.07.2026 10:56:40,Szücs Magdalena,Online schůzka s klientem,14,2026-07-02 10:56:40,2026-07-02,2026-07-02 11:10:40,Praha 2 (Jugoslávská),20,8,480
3,459,6,02.07.2026 11:41:50,Tauchman Dominik,Online schůzka s klientem,46,2026-07-02 11:41:50,2026-07-02,2026-07-02 12:27:50,Praha 2 (Jugoslávská),20,8,480
4,459,18,02.07.2026 12:43:56,Radil Ivo,Online schůzka s klientem,60,2026-07-02 12:43:56,2026-07-02,2026-07-02 13:43:56,Praha 2 (Jugoslávská),20,8,480


## 3. Výpočet vytíženosti

- **`workstation_daily`** — denní vytíženost každého jednotlivého pracoviště (vůči kapacitě 1 pracoviště).
- **`branch_daily`** — denní vytíženost celé pobočky (součet všech pracovišť vůči `NO_WPL × CAPACITY`).
- Dny bez zaznamenané aktivity jsou dopočítány jako 0 % (pracoviště bylo otevřené, ale nevyužité) — nejsou tedy
  z průměru tiše vynechány.

In [6]:
workstation_daily = rl.compute_workstation_daily(merged)
branch_daily = rl.compute_branch_daily(merged)

branch_summary = rl.summarize_branches(branch_daily, workspaces)
branch_summary

,BRANCH_ID,BRANCH_NAME,NO_WORKSTATIONS,CAPACITY_HOURS,PRUMERNA_VYTIZENOST_PCT,MAX_VYTIZENOST_PCT,DNI_KRITICKA,CELKEM_HODIN,POCET_DNI,BUCKET
0,459,Praha 2 (Jugoslávská),20,8,5.791667,5.791667,0,9.266667,1,Nízká
1,174,Chomutov,3,8,NaN,NaN,0,0.000000,0,Bez dat
2,148,Mladá Boleslav,5,8,NaN,NaN,0,0.000000,0,Bez dat
3,160,Česká Lípa,4,8,NaN,NaN,0,0.000000,0,Bez dat
4,228,Kolín,4,8,NaN,NaN,0,0.000000,0,Bez dat
5,266,Jičín,4,8,NaN,NaN,0,0.000000,0,Bez dat
6,235,Kutná Hora,2,8,NaN,NaN,0,0.000000,0,Bez dat
7,60,Zlín,6,8,NaN,NaN,0,0.000000,0,Bez dat
8,107,Uherské Hradiště,3,8,NaN,NaN,0,0.000000,0,Bez dat
9,114,Vyškov,2,8,NaN,NaN,0,0.000000,0,Bez dat


## 4. Vytíženost poboček — přehled

In [7]:
fig = rl.fig_branch_utilization_bar(branch_summary)
fig.show()

## 5. Kapacita: přibývají pracoviště?

Kontrola, zda se na některé pobočce reálně využívá víc pracovišť (unikátní `PRACOVISTE_ID` v datech),
než je aktuálně registrováno v `work_spaces.xlsx` — signál, že evidenci kapacity je třeba navýšit.

In [8]:
growth_flags = rl.compute_capacity_growth_flags(merged, workspaces)
growth_flags

,BRANCH_ID,BRANCH_NAME,POUZITA_PRACOVISTE,NO_WORKSTATIONS,ROZDIL,PREKROCENO
0,459,Praha 2 (Jugoslávská),5,20,-15,False


## 6. Vytíženost jednotlivých pracovišť (heatmapa den × pracoviště)

In [9]:
for branch_id in sorted(workstation_daily["BRANCH_ID"].unique()):
    name = workstation_daily.loc[workstation_daily["BRANCH_ID"] == branch_id, "BRANCH_NAME"].iloc[0]
    rl.fig_workstation_heatmap(workstation_daily, branch_id, name).show()

## 7. Trend vytíženosti v čase

In [10]:
if branch_daily["DATE"].nunique() > 1:
    rl.fig_utilization_trend(branch_daily).show()
else:
    print("K dispozici je jen jeden den dat — trend v čase se zobrazí, jakmile bude dat víc.")

K dispozici je jen jeden den dat — trend v čase se zobrazí, jakmile bude dat víc.


## 8. Skladba aktivit a nejvytíženější zaměstnanci

In [11]:
activity_breakdown = rl.compute_activity_breakdown(merged)
employee_summary = rl.compute_employee_summary(merged)

rl.fig_activity_mix(activity_breakdown).show()
rl.fig_employee_top(employee_summary).show()

## 9. Kontrola kolizí rezervací

Zjednodušená kontrola: hledá sousedící (dle času začátku) rezervace stejného pracoviště, které se
časově překrývají.

In [12]:
overlaps = rl.detect_possible_overlaps(merged)
overlaps

,BRANCH_ID,BRANCH_NAME,WORKSTATION_ID,PREDCHOZI_ZAMESTNANEC,PREDCHOZI_START,PREDCHOZI_END,ZAMESTNANEC,START,END,ACTIVITY
0,459,Praha 2 (Jugoslávská),5,Škábová Alena,2026-07-02 16:06:34,2026-07-02 16:51:34,Škábová Alena,2026-07-02 16:07:05,2026-07-02 16:23:05,Telefonát s klientem
1,459,Praha 2 (Jugoslávská),5,Škábová Alena,2026-07-02 16:07:05,2026-07-02 16:23:05,Škábová Alena,2026-07-02 16:07:28,2026-07-02 16:17:28,Telefonát s klientem


## 10. Generování finálního HTML reportu

In [13]:
report_path = rl.build_html_report(
    OUTPUT_HTML,
    period_start=branch_daily["DATE"].min(),
    period_end=branch_daily["DATE"].max(),
    activities=activities,
    workspaces=workspaces,
    merged=merged,
    branch_summary=branch_summary,
    branch_daily=branch_daily,
    workstation_daily=workstation_daily,
    growth_flags=growth_flags,
    activity_breakdown=activity_breakdown,
    employee_summary=employee_summary,
    overlaps=overlaps,
    data_issues=data_issues,
    unknown_branches=unknown_branches,
)
print(f"Report vygenerován: {report_path.resolve()}")

Report vygenerován: /home/user/bo_online_report/output/vytizenost_report.html


Report lze otevřít přímo v prohlížeči (dvojklik na soubor v `output/`), nebo jej zobrazit zde v notebooku:

In [14]:
from IPython.display import IFrame
IFrame(src=str(report_path), width="100%", height=800)